# 🌱 AgriMesh — Train the crop-disease model (Colab)

**Runtime → Change runtime type → GPU** before you start.

Self-contained: you only upload the dataset zip. The notebook normalizes the
134 messy folders → 107 clean classes, trains MobileNetV2, and exports a
~4MB TensorFlow.js model that drops straight into `frontend/models/`.

Run the cells top to bottom.

## 1. Check GPU

In [ ]:
import tensorflow as tf
print('TF', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus or 'NONE — set Runtime > Change runtime type > GPU!')

## 2. Install the TF.js converter
We force **legacy Keras (Keras 2)** — the TensorFlow.js converter is reliable
with it and flaky with Keras 3. Ignore pip dependency warnings.

In [ ]:
!pip install -q tensorflowjs tf_keras
print('installed — RESTART NOT NEEDED, just continue')

## 3. Get the dataset
**Recommended:** upload `archive (1).zip` to your Google Drive (My Drive root),
then run this cell. It mounts Drive, unzips into `/content`, and auto-finds the
`dataset_clean_final` folder wherever it lands.

_No Drive? Comment the mount lines and use the `files.upload()` fallback at the bottom._

In [ ]:
import os, glob, zipfile
from google.colab import drive
drive.mount('/content/drive')

# EDIT if your zip has a different name / location:
ZIP = '/content/drive/MyDrive/archive (1).zip'

if not glob.glob('/content/**/dataset_clean_final', recursive=True):
    print('unzipping', ZIP)
    with zipfile.ZipFile(ZIP) as z: z.extractall('/content/data')

hits = glob.glob('/content/**/dataset_clean_final', recursive=True)
assert hits, 'dataset_clean_final not found — check ZIP path/name'
DATA = hits[0]
print('DATA =', DATA, '|', len(os.listdir(DATA)), 'raw folders')

# --- fallback (no Drive): uncomment ---
# from google.colab import files; up = files.upload()
# with zipfile.ZipFile(list(up)[0]) as z: z.extractall('/content/data')
# DATA = glob.glob('/content/**/dataset_clean_final', recursive=True)[0]

## 4. Normalize labels → 107 clean classes
Same logic as `model/normalize.py`: collapse `_`/`__`/`___`, strip repeated crop
prefixes, fix spelling variants (Gauva→Guava, etc.). The 27 duplicate folder
groups merge into one label each, so training isn't punished for picking a twin.

In [ ]:
import re, csv, json, collections

CROP_ALIAS = {'gauva': 'guava'}
COND_ALIAS = {
    'yellowleaf_curl_virus': 'yellow_leaf_curl_virus',
    'tomato_yellowleaf_curl_virus': 'yellow_leaf_curl_virus',
    'tomato_mosaic_virus': 'mosaic_virus',
    'spider_mites_two_spotted_spider_mite': 'spider_mites',
    'spider_mites': 'spider_mites',
}
def canon(folder):
    s = re.sub(r'[()]', '', folder.lower())
    s = re.sub(r'_+', '_', s).strip('_')
    parts = s.split('_')
    crop = CROP_ALIAS.get(parts[0], parts[0])
    cond = parts[1:]
    while cond and cond[0] in (crop, 'tomato'): cond = cond[1:]
    cond = '_'.join(cond) or 'unknown'
    return crop, COND_ALIAS.get(cond, cond)

folders = sorted(d for d in os.listdir(DATA) if os.path.isdir(os.path.join(DATA, d)))
groups = {}
for f in folders: groups.setdefault(canon(f), []).append(f)
classes = sorted(groups)
B36 = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
code_of = lambda i: 'D' + B36[i // 36] + B36[i % 36]

# labels.json — index i (model output) -> disease + D-code. MUST match frontend.
labels = []
for i, (crop, cond) in enumerate(classes):
    labels.append({'code': code_of(i), 'canonical': f'{crop}__{cond}', 'crop': crop,
                   'condition': cond, 'label': f"{crop.title()} - {cond.replace('_',' ').title()}",
                   'healthy': cond == 'healthy'})
json.dump(labels, open('/content/labels.json','w'), ensure_ascii=False, indent=1)

# (path, class_index) list — 27 duplicate folders feed the SAME index
idx_of = {c: i for i, c in enumerate(classes)}
paths, ys = [], []
for (crop, cond), raws in groups.items():
    for raw in raws:
        d = os.path.join(DATA, raw)
        for fn in os.listdir(d): paths.append(os.path.join(d, fn)); ys.append(idx_of[(crop, cond)])
NUM = len(classes)
print(f'{len(folders)} raw folders -> {NUM} classes, {len(paths)} images')

## 5. Build the tf.data pipeline
Cache + prefetch (GPU-bound not IO-bound), augmentation on train only, and
**class weights** so the tiny classes (Soybean mosaic = 22 imgs) aren't ignored
next to the big ones (Tomato healthy = 1539).

In [ ]:
import numpy as np
IMG, BATCH, SEED = 224, 32, 42

freq = collections.Counter(ys); total = len(ys)
class_weight = {i: total / (NUM * freq[i]) for i in range(NUM)}

def decode(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.image.resize(img, [IMG, IMG]), label

aug = tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'),
                           tf.keras.layers.RandomRotation(0.1),
                           tf.keras.layers.RandomZoom(0.1)])

full = tf.data.Dataset.from_tensor_slices((paths, ys)).shuffle(len(paths), seed=SEED)
val_n = int(0.15 * len(paths))
def prep(ds, training):
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if training: ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
val_ds = prep(full.take(val_n), False)
train_ds = prep(full.skip(val_n), True)
print('train/val split ready')

## 6. Build MobileNetV2
ImageNet preprocessing is **baked into the graph**, so the browser only feeds raw
0–255 pixels — no normalization bug possible on the frontend.

In [ ]:
tf.keras.mixed_precision.set_global_policy('mixed_float16')
base = tf.keras.applications.MobileNetV2((IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False
inp = tf.keras.Input((IMG, IMG, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inp)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(NUM, activation='softmax', dtype='float32')(x)
model = tf.keras.Model(inp, out)
cb = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
      tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3)]
model.summary()

## 7. Phase 1 — train the head (frozen base)

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h1 = model.fit(train_ds, validation_data=val_ds, epochs=15,
               class_weight=class_weight, callbacks=cb)

## 8. Phase 2 — fine-tune the top of the base
Unfreeze the last 30 layers at a low LR for the final accuracy push.

In [ ]:
base.trainable = True
for l in base.layers[:-30]: l.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=10,
               class_weight=class_weight, callbacks=cb)
print('best val acc:', round(max(h2.history['val_accuracy']), 4))

## 9. Sanity-check on a validation batch

In [ ]:
import matplotlib.pyplot as plt
xb, yb = next(iter(val_ds))
pred = model.predict(xb).argmax(1)
plt.figure(figsize=(12, 6))
for i in range(min(8, len(xb))):
    plt.subplot(2, 4, i+1); plt.imshow(xb[i].numpy().astype('uint8')); plt.axis('off')
    ok = pred[i] == yb[i].numpy()
    plt.title(('✓' if ok else '✗') + ' ' + labels[pred[i]]['label'][:18], color='green' if ok else 'red', fontsize=8)
plt.tight_layout(); plt.show()

## 10a. Export the trained model to disk (run FIRST)
We dump the in-memory model to a TF SavedModel **before** installing the converter,
so nothing is lost if pip changes a dependency. Also backs up to Drive.

In [ ]:
model.export('/content/sm')                                    # Keras 3 -> TF SavedModel
model.save('/content/drive/MyDrive/agrimesh_model.keras')      # Drive backup (survives restart)
print('exported ✓  — now run 10b')

## 10b. Convert to TensorFlow.js + download
The converter runs as a separate CLI process reading from disk, so the Keras 2/3
version drama can't touch it. Produces `web_model.zip` (model.json + *.bin + labels.json).
**Unzip its contents into `frontend/models/`** — the app auto-detects it, no code change.

In [ ]:
!pip install -q tensorflowjs
!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model /content/sm /content/web_model
import os, shutil
shutil.copy('/content/labels.json', '/content/web_model/labels.json')
shutil.make_archive('/content/web_model', 'zip', '/content/web_model')
size = sum(os.path.getsize('/content/web_model/' + f) for f in os.listdir('/content/web_model')) / 1e6
print(f'files: {os.listdir("/content/web_model")}\nmodel size: {size:.1f} MB')
from google.colab import files
files.download('/content/web_model.zip')